# Company Name / Alias Extraction — Databricks PySpark

**Goal:** Read a CSV containing `Co Code` and a company description; extract the primary, former and related company names; preserve each name's original words, spelling, punctuation and casing; persist the resulting data in Unity Catalog (Delta tables).

**No text substitutions:** `&` does **not** become `AND`, `PVT` does **not** become `PRIVATE`, and `LTD` does **not** become `LIMITED`. Only relationship phrases, known non-name annotations such as `(DNS)`, excess whitespace, and stray enclosing brackets are removed from extracted aliases.

**Outputs** (full-snapshot overwrite):
- `company_master_raw`: source rows, with ingestion metadata.
- `company_alias_master`: one row per `(Co Code, company name, alias type)`.
- `company_alias_review`: ambiguous extracted names and invalid source records requiring review.

> **Important:** `RELATED_ENTITY` means the CSV associates an entity with a Co Code; it does **not** establish that it is the same legal company or employer. This notebook prepares a reference table; it does not classify salary transactions.

## 1. Configuration

Change the catalog, schema, Volume CSV path, and source column names before running. The Unity Catalog catalog/schema must exist and the job identity needs read/write permissions.

In [ ]:
import re
from pyspark.sql import functions as F

# ===== EDIT THESE SETTINGS =====
CATALOG = "your_catalog"
SCHEMA = "your_schema"
CSV_PATH = "/Volumes/your_catalog/your_schema/your_volume/company.csv"

CODE_COLUMN = "Co Code"          # Exact CSV header
NAME_COLUMN = "Company Name"     # Exact CSV header

RAW_TABLE = f"{CATALOG}.{SCHEMA}.company_master_raw"
ALIAS_TABLE = f"{CATALOG}.{SCHEMA}.company_alias_master"
REVIEW_TABLE = f"{CATALOG}.{SCHEMA}.company_alias_review"

# Full CSV snapshot: replace the three destination tables on each run.
# Do not switch to append without adding deduplication / MERGE logic.
WRITE_MODE = "overwrite"

# Validate identifiers before interpolating them into SQL/table names.
for identifier in (CATALOG, SCHEMA):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Unsupported catalog/schema identifier: {identifier!r}")

print(f"Source: {CSV_PATH}")
print(f"Raw table: {RAW_TABLE}")
print(f"Alias table: {ALIAS_TABLE}")
print(f"Review table: {REVIEW_TABLE}")

## 2. Extraction rules

The rules recognize explicit relationship phrases, **not** `AND`, `&`, commas, slashes or legal suffixes. Add further patterns here after reviewing actual source examples. Rule order matters: former-name phrases are recognized before the generic `KNOWN AS` pattern.

In [ ]:
# ASCII Record Separator used internally as a delimiter (unlikely in company text).
SEPARATOR = "\x1e"

# Each matched phrase becomes: [delimiter]ALIAS_TYPE[delimiter]
ALIAS_RULES = [
    (
        "RELATED_ENTITY",
        r"(?i)(?:\s*[\(\[\{]+\s*)*\b(?:ALSO|ASLO)\s+INCLUDES?\b\s*:?\s*"
    ),
    (
        "FORMER",
        r"(?i)(?:\s*[\(\[\{]+\s*)*\b(?:EARLIER|FORMERLY|PREVIOUSLY|WAS)"
        r"\s+(?:(?:KNOWN|KNOW)\s+AS|CALLED|NAMED)\b\s*:?\s*"
    ),
    (
        "FORMER",
        r"(?i)(?:\s*[\(\[\{]+\s*)*\b(?:OLD|PREVIOUS)\s+NAME\b\s*[:\-] ?"
    ),
    (
        "AKA",
        r"(?i)(?:\s*[\(\[\{]+\s*)*\b(?:ALSO\s+KNOWN\s+AS|KNOWN\s+AS|KNOW\s+AS|TRADING\s+AS)\b\s*:?\s*"
    ),
]

# Annotations that your source confirms are NOT part of a company name.
# Keep conservative: never blindly delete arbitrary text inside parentheses.
ANNOTATION_REGEX = r"(?i)\(\s*DNS\s*\)"

UNPARSED_RELATION_REGEX = (
    r"(?i)\b(?:ASLO\s+INCLUDES?|ALSO\s+INCLUDES?|KNOWN\s+AS|KNOW\s+AS|"
    r"FORMERLY|PREVIOUSLY|EARLIER|OLD\s+NAME|TRADING\s+AS)\b"
)

print("Alias extraction rules:", len(ALIAS_RULES))

## 3. Extraction function

This function uses native Spark expressions (no row-by-row Python UDF). It retains the full original source description for audit purposes. The `parse_status` flag identifies records requiring manual review.

In [ ]:
def extract_company_aliases(valid_source_df):
    """Return extracted company aliases from validated source rows.

    Required columns: co_code, company_name_raw, source_record_id, ingested_at.
    No uppercase conversion, punctuation replacement or legal-suffix expansion.
    """
    marked = (
        valid_source_df
        .withColumn(
            "_marked",
            F.regexp_replace(F.col("company_name_raw"), ANNOTATION_REGEX, " ")
        )
        # Convert CSV line breaks and tabs to spaces (not the company wording).
        .withColumn("_marked", F.regexp_replace("_marked", r"[\r\n\t]", " "))
    )

    for alias_type, pattern in ALIAS_RULES:
        marked = marked.withColumn(
            "_marked",
            F.regexp_replace("_marked", pattern, SEPARATOR + alias_type + SEPARATOR)
        )

    pieces_df = (
        marked
        .withColumn("_pieces", F.split(F.col("_marked"), SEPARATOR))
        .select(
            "co_code", "company_name_raw", "source_record_id", "ingested_at",
            "_pieces", F.posexplode("_pieces").alias("_idx", "_piece")
        )
    )

    # Index 0 is primary; indices 2, 4, ... are company aliases.
    # Indices 1, 3, ... are the relationship labels.
    alias_rows = (
        pieces_df
        .filter((F.col("_idx") % 2) == 0)
        .withColumn(
            "alias_type",
            F.when(F.col("_idx") == 0, F.lit("PRIMARY"))
             .otherwise(F.element_at(F.col("_pieces"), F.col("_idx")))
        )
    )

    # Only strip brackets functioning as wrappers at the segment boundaries.
    # Keep meaningful internal brackets: "ABC (INDIA) PVT LTD" remains unchanged.
    company = F.trim(F.col("_piece"))
    company = F.regexp_replace(company, r"^[\(\[\{]+\s*", "")
    company = F.regexp_replace(company, r"\s*[\(\[\{]+\s*$", "")
    for opener, closing_pattern in [
        ("(", r"\)+\s*$"),
        ("[", r"\]+\s*$"),
        ("{", r"\}+\s*$"),
    ]:
        company = F.when(
            F.instr(company, opener) == 0,
            F.regexp_replace(company, closing_pattern, "")
        ).otherwise(company)

    # Spaces are not company-name tokens. No words/punctuation are rewritten.
    company = F.trim(F.regexp_replace(company, r"\s+", " "))

    aliases = (
        alias_rows
        .withColumn("company_name", company)
        .filter(F.length("company_name") > 0)
    )

    # Data quality checks: leftover relationship phrases, very short names,
    # and unmatched brackets after the extraction.
    opening_parens = (
        F.length("company_name") -
        F.length(F.regexp_replace("company_name", r"\(", ""))
    )
    closing_parens = (
        F.length("company_name") -
        F.length(F.regexp_replace("company_name", r"\)", ""))
    )
    opening_sq = (
        F.length("company_name") -
        F.length(F.regexp_replace("company_name", r"\[", ""))
    )
    closing_sq = (
        F.length("company_name") -
        F.length(F.regexp_replace("company_name", r"\]", ""))
    )
    unbalanced = ((opening_parens != closing_parens) | (opening_sq != closing_sq))

    aliases = (
        aliases
        .withColumn(
            "parse_issue",
            F.concat_ws(
                "; ",
                F.when(F.col("company_name").rlike(UNPARSED_RELATION_REGEX),
                       "UNPARSED_RELATION_PHRASE"),
                F.when(F.length("company_name") < 3, "VERY_SHORT_NAME"),
                F.when(unbalanced, "UNBALANCED_BRACKETS")
            )
        )
        .withColumn(
            "parse_status",
            F.when(F.length("parse_issue") > 0, "REVIEW")
             .otherwise("READY")
        )
        .select(
            "co_code", "company_name", "alias_type", "parse_status",
            "parse_issue", "company_name_raw", "source_record_id", "ingested_at"
        )
        .dropDuplicates(["source_record_id", "company_name", "alias_type"])
    )

    return aliases

## 4. Self-test against representative descriptions

Run this before writing tables. It specifically verifies that `&`, `PVT`, `LTD`, `Ltd.`, misspellings and meaningful internal brackets remain untouched, while the former and related names are extracted.

In [ ]:
sample_records = [
    ("1001", "Arcum Engineering Ltd."),
    ("1002", "ABC & SONS PVT LTD (DNS)"),
    ("1003", "ADITYA BIRLA FINANCE LIMITED ASLO INCLUDES BIRLA INSURANCE ADVISORY AND BROKING SERVICES LIMITED (( Earlier know as BIRLA GLOBAL FINANCE LTD ( WAS KNOW AS BIRLA GLOBAL FINANCE COMPNAY LTD)"),
    ("1004", "XYZ TECHNOLOGIES LTD (Formerly known as XYZ SOFTWARE PVT LTD)"),
    ("1005", "ABC (INDIA) PVT LTD"),
    ("1006", "GLOBAL TECH LTD ALSO KNOWN AS GLOBAL TECHNOLOGIES"),
    ("1007", "MEGA CORP LTD [Previously known as OLD MEGA PVT LTD]"),
]

sample_df = (
    spark.createDataFrame(sample_records, ["co_code", "company_name_raw"])
    .withColumn("source_record_id", F.sha2(F.to_json(F.struct("co_code", "company_name_raw")), 256))
    .withColumn("ingested_at", F.current_timestamp())
)

sample_results_df = extract_company_aliases(sample_df)
actual = {
    (r["co_code"], r["company_name"], r["alias_type"])
    for r in sample_results_df.select("co_code", "company_name", "alias_type").collect()
}
expected = {
    ("1001", "Arcum Engineering Ltd.", "PRIMARY"),
    ("1002", "ABC & SONS PVT LTD", "PRIMARY"),
    ("1003", "ADITYA BIRLA FINANCE LIMITED", "PRIMARY"),
    ("1003", "BIRLA INSURANCE ADVISORY AND BROKING SERVICES LIMITED", "RELATED_ENTITY"),
    ("1003", "BIRLA GLOBAL FINANCE LTD", "FORMER"),
    ("1003", "BIRLA GLOBAL FINANCE COMPNAY LTD", "FORMER"),
    ("1004", "XYZ TECHNOLOGIES LTD", "PRIMARY"),
    ("1004", "XYZ SOFTWARE PVT LTD", "FORMER"),
    ("1005", "ABC (INDIA) PVT LTD", "PRIMARY"),
    ("1006", "GLOBAL TECH LTD", "PRIMARY"),
    ("1006", "GLOBAL TECHNOLOGIES", "AKA"),
    ("1007", "MEGA CORP LTD", "PRIMARY"),
    ("1007", "OLD MEGA PVT LTD", "FORMER"),
}

missing = expected - actual
unexpected = actual - expected
assert not missing, f"Missing expected aliases: {missing}"
assert not unexpected, f"Unexpected extracted aliases: {unexpected}"
print(f"Self-test passed: {len(expected)} exact alias records")
display(sample_results_df.orderBy("co_code", "alias_type"))

## 5. Read CSV, validate headers, and persist raw company master

The raw table preserves the CSV values. For reproducibility, this notebook assumes each CSV is a **full current snapshot**, so writes use `overwrite`. If the source is incremental, implement a Delta `MERGE` instead.

In [ ]:
csv_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")  # Keep Co Code as a string (preserves leading zeros)
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "FAILFAST")
    .load(CSV_PATH)
)

missing_headers = [c for c in (CODE_COLUMN, NAME_COLUMN) if c not in csv_df.columns]
if missing_headers:
    raise ValueError(
        f"Missing CSV column(s): {missing_headers}. CSV headers: {csv_df.columns}"
    )

raw_df = (
    csv_df.select(
        F.col(CODE_COLUMN).cast("string").alias("co_code"),
        F.col(NAME_COLUMN).cast("string").alias("company_name_raw")
    )
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn(
        "source_record_id",
        F.sha2(F.to_json(F.struct("co_code", "company_name_raw")), 256)
    )
    .dropDuplicates(["source_record_id"])
)

# The delimiter must not occur in the source data.
if raw_df.filter(F.instr(F.col("company_name_raw"), SEPARATOR) > 0).limit(1).count():
    raise ValueError("Source CSV contains the internal extraction delimiter.")

# Persist raw snapshot
(
    raw_df.write.format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_TABLE)
)

print("Raw rows:", spark.table(RAW_TABLE).count())

## 6. Extract aliases, validate, and persist the alias master

Valid rows must have a non-empty Co Code and description. Every identified alias retains its source Co Code and its original full company description.

In [ ]:
raw_saved = spark.table(RAW_TABLE)

valid_source_df = (
    raw_saved
    .filter(F.length(F.trim(F.col("co_code"))) > 0)
    .filter(F.length(F.trim(F.col("company_name_raw"))) > 0)
    .withColumn("co_code", F.trim(F.col("co_code")))
)

alias_df = extract_company_aliases(valid_source_df)

# Deduplicate identical aliases for each Co Code, retaining their alias types.
# Distinct source descriptions remain in company_name_raw for audit purposes.
(
    alias_df.write.format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(ALIAS_TABLE)
)

print("Alias rows:", spark.table(ALIAS_TABLE).count())
display(spark.table(ALIAS_TABLE).orderBy("co_code", "alias_type", "company_name"))

## 7. Persist review/exception records

Records containing still-unparsed relationship phrases, suspicious brackets, or missing source fields are **not silently discarded**. Review `company_alias_review`, then extend `ALIAS_RULES` / `ANNOTATION_REGEX` as required.

In [ ]:
bad_raw_df = (
    raw_saved
    .filter(
        (F.col("co_code").isNull()) |
        (F.length(F.trim(F.col("co_code"))) == 0) |
        (F.col("company_name_raw").isNull()) |
        (F.length(F.trim(F.col("company_name_raw"))) == 0)
    )
    .select(
        "co_code", "company_name_raw", "source_record_id", "ingested_at"
    )
    .withColumn("company_name", F.lit(None).cast("string"))
    .withColumn("alias_type", F.lit(None).cast("string"))
    .withColumn("parse_status", F.lit("REVIEW"))
    .withColumn("parse_issue", F.lit("EMPTY_CODE_OR_COMPANY_NAME"))
)

review_alias_df = (
    alias_df.filter(F.col("parse_status") == "REVIEW")
)

# Also report valid source records for which no company name was extracted.
no_alias_df = (
    valid_source_df
    .join(
        alias_df.select("source_record_id").distinct(),
        on="source_record_id",
        how="left_anti"
    )
    .withColumn("company_name", F.lit(None).cast("string"))
    .withColumn("alias_type", F.lit(None).cast("string"))
    .withColumn("parse_status", F.lit("REVIEW"))
    .withColumn("parse_issue", F.lit("NO_COMPANY_NAME_EXTRACTED"))
)

review_df = (
    review_alias_df
    .unionByName(bad_raw_df)
    .unionByName(no_alias_df)
    .select(
        "co_code", "company_name", "alias_type", "parse_issue",
        "parse_status", "company_name_raw", "source_record_id", "ingested_at"
    )
)

(
    review_df.write.format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(REVIEW_TABLE)
)

print("Review rows:", spark.table(REVIEW_TABLE).count())
display(spark.table(REVIEW_TABLE))

## 8. Final quality summary and sample SQL

Names with `parse_status = 'READY'` have passed these extraction checks; this is **not** an independent legal-entity verification. Related names should be treated as candidate references rather than guaranteed synonyms when matching transaction narrations.

In [ ]:
display(
    spark.table(ALIAS_TABLE)
    .groupBy("alias_type", "parse_status")
    .count()
    .orderBy("alias_type", "parse_status")
)

print("Tables created:")
for table_name in (RAW_TABLE, ALIAS_TABLE, REVIEW_TABLE):
    print(" -", table_name)

# Example: aliases for one Co Code
# display(spark.table(ALIAS_TABLE).filter(F.col("co_code") == "1003"))

### Example SQL (run in a Databricks SQL cell)

```sql
SELECT co_code, company_name, alias_type, parse_status
FROM your_catalog.your_schema.company_alias_master
WHERE co_code = '1003'
ORDER BY alias_type, company_name;
```

**Limitations:** Without explicit relationship wording, a generic `( ... )` note is deliberately *not* split into a separate legal-company name, because it could be part of an ordinary name (for example `ABC (INDIA) PVT LTD`). Add vetted parsing rules for additional patterns observed in your CSV. The `company_alias_master` is the prepared reference dataset; transaction matching belongs in a separate job.